# 02 - Data Validation

**Objective:** Validate wine records using Pydantic models and Pandera DataFrame schemas.

**Tools:** Pydantic (row-level), Pandera (DataFrame-level)

**Steps:**
1. Define a Pydantic model for wine records
2. Implement row-level validation
3. Define a Pandera DataFrame schema
4. Implement DataFrame-level validation
5. Test on clean and corrupted data

In [1]:
import sys
from pathlib import Path

# Add wine_execise/ to the path so local modules (data_validation, config, etc.) are importable
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd

from data_validation import WineRecord, WineSchema
from data_validation import validate_with_pydantic, validate_with_pandera

print("Libraries imported successfully")


Libraries imported successfully


In [2]:
# Load the clean data from data/processed/clean_data.csv
# We'll validate this data using both Pydantic and Pandera.

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Data shape: (178, 14)
Columns: ['class', 'alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280_od315', 'proline']


### Pydantic Model Definition

Pydantic validates data at the **record level** — each row is checked individually.
Define a model with typed fields and add `@field_validator` decorators to enforce
business rules like valid class labels, positive measurements, etc.

This catches row-level issues like a class value of -1, a negative Alcohol,
or a proline of 10000 that might slip through type-only checks.

In [3]:
# WineRecord is imported from data_validation.schemas.
# It validates each wine record at the row level with domain-aware constraints.

print("WineRecord fields:")
for name, field in WineRecord.model_fields.items():
    print(f"  {name}: {field.annotation}")


WineRecord fields:
  alcohol: <class 'float'>
  malic_acid: <class 'float'>
  ash: <class 'float'>
  alcalinity_of_ash: <class 'float'>
  magnesium: <class 'int'>
  total_phenols: <class 'float'>
  flavanoids: <class 'float'>
  nonflavanoid_phenols: <class 'float'>
  proanthocyanins: <class 'float'>
  color_intensity: <class 'float'>
  hue: <class 'float'>
  od280_od315: <class 'float'>
  proline: <class 'int'>
  class_: <class 'int'>


In [4]:
# Test Pydantic row-level validation on clean data
valid_df, invalid_df = validate_with_pydantic(df)
print(f"Valid rows:   {len(valid_df)}")
print(f"Invalid rows: {len(invalid_df)}")

if not invalid_df.empty:
    print("\nSample invalid rows:")
    print(invalid_df[["_validation_error"]].head())


2026-05-31 16:52:17,493 - data_validation.validation - INFO - Pydantic validation: 178 valid, 0 invalid (out of 178)


Valid rows:   178
Invalid rows: 0


### Pandera Schema Definition

Pandera validates data at the **DataFrame level** — instead of checking one row at a time,
it defines column-wide constraints (data types, value ranges, nullable rules, etc.).

This is more efficient for bulk validation and catches issues like a column
that unexpectedly contains strings instead of numbers.

In [5]:
# WineSchema is imported from data_validation.schemas.
# It validates the entire DataFrame at once using Pandera column-level checks.

print("WineSchema columns:")
for col_name, col in WineSchema.columns.items():
    print(f"  {col_name}: dtype={col.dtype}, nullable={col.nullable}")


WineSchema columns:
  alcohol: dtype=float64, nullable=False
  malic_acid: dtype=float64, nullable=False
  ash: dtype=float64, nullable=False
  alcalinity_of_ash: dtype=float64, nullable=False
  magnesium: dtype=int64, nullable=False
  total_phenols: dtype=float64, nullable=False
  flavanoids: dtype=float64, nullable=False
  nonflavanoid_phenols: dtype=float64, nullable=False
  proanthocyanins: dtype=float64, nullable=False
  color_intensity: dtype=float64, nullable=False
  hue: dtype=float64, nullable=False
  od280_od315: dtype=float64, nullable=False
  proline: dtype=int64, nullable=False
  class: dtype=int64, nullable=False


In [6]:
# Test Pandera DataFrame-level validation on clean data
passed, error_msg = validate_with_pandera(df)
print(f"Pandera: {'PASSED' if passed else 'FAILED'}")
if error_msg:
    print(f"\nErrors:\n{error_msg}")


2026-05-31 16:52:17,578 - data_validation.validation - INFO - Pandera validation: PASSED


Pandera: PASSED


In [7]:
# Test both validators on each pre-generated corrupted dataset
CORRUPTED_DIR = Path("../data/processed/corrupted")

for corrupted_file in sorted(CORRUPTED_DIR.glob("*.csv")):
    df_corrupted = pd.read_csv(corrupted_file)
    print(f"\n--- {corrupted_file.name} ---")

    valid_df, invalid_df = validate_with_pydantic(df_corrupted)
    print(f"  Pydantic: {len(valid_df)} valid, {len(invalid_df)} invalid")

    passed, error_msg = validate_with_pandera(df_corrupted)
    print(f"  Pandera:  {'PASSED' if passed else 'FAILED'}")


2026-05-31 16:52:17,602 - data_validation.validation - INFO - Pydantic validation: 176 valid, 2 invalid (out of 178)


2026-05-31 16:52:17,614 - data_validation.validation - WARNING - Pandera validation: FAILED ({
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "alcohol",
                "check": "in_range(8, 17)",
                "error": "Column 'alcohol' failed element-wise validator number 0: in_range(8, 17) failure cases: 7.986999999999999, 7.720999999999999"
            }
        ]
    }
})
2026-05-31 16:52:17,625 - data_validation.validation - INFO - Pydantic validation: 222 valid, 0 invalid (out of 222)
2026-05-31 16:52:17,632 - data_validation.validation - INFO - Pandera validation: PASSED
2026-05-31 16:52:17,641 - data_validation.validation - INFO - Pydantic validation: 186 valid, 0 invalid (out of 186)
2026-05-31 16:52:17,648 - data_validation.validation - INFO - Pandera validation: PASSED



--- corrupted_bias.csv ---
  Pydantic: 176 valid, 2 invalid
  Pandera:  FAILED

--- corrupted_duplicates_heavy.csv ---
  Pydantic: 222 valid, 0 invalid
  Pandera:  PASSED

--- corrupted_duplicates_light.csv ---
  Pydantic: 186 valid, 0 invalid
  Pandera:  PASSED

--- corrupted_missing_heavy.csv ---


2026-05-31 16:52:17,656 - data_validation.validation - INFO - Pydantic validation: 143 valid, 35 invalid (out of 178)


  Pydantic: 143 valid, 35 invalid


...

  Pandera:  FAILED

--- corrupted_missing_light.csv ---


2026-05-31 16:52:17,680 - data_validation.validation - INFO - Pydantic validation: 170 valid, 8 invalid (out of 178)
2026-05-31 16:52:17,694 - data_validation.validation - WARNING - Pandera validation: FAILED ({
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "class",
                "check": "not_nullable",
                "error": "non-nullable series 'class' contains null values:16    NaN19    NaN30    NaN45    NaN67    NaN119   NaN140   NaN174   NaNName: class, dtype: float64"
            }
        ],
        "WRONG_DATATYPE": [
            {
                "schema": null,
                "column": "class",
                "check": "dtype('int64')",
                "error": "expected series 'class' to have type int64, got float64"
            }
        ]
    }
})


  Pydantic: 170 valid, 8 invalid
  Pandera:  FAILED

--- corrupted_noise_high.csv ---


2026-05-31 16:52:17,703 - data_validation.validation - INFO - Pydantic validation: 178 valid, 0 invalid (out of 178)
2026-05-31 16:52:17,709 - data_validation.validation - INFO - Pandera validation: PASSED
2026-05-31 16:52:17,717 - data_validation.validation - INFO - Pydantic validation: 178 valid, 0 invalid (out of 178)
2026-05-31 16:52:17,724 - data_validation.validation - INFO - Pandera validation: PASSED
2026-05-31 16:52:17,732 - data_validation.validation - INFO - Pydantic validation: 173 valid, 5 invalid (out of 178)
2026-05-31 16:52:17,758 - data_validation.validation - WARNING - Pandera validation: FAILED ({
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "alcohol",
                "check": "in_range(8, 17)",
                "error": "Column 'alcohol' failed element-wise validator number 0: in_range(8, 17) failure cases: 37.02, 35.46, 37.53, 37.26, 36.75"
            },
            {
                "schema": nu

  Pydantic: 178 valid, 0 invalid
  Pandera:  PASSED

--- corrupted_noise_low.csv ---
  Pydantic: 178 valid, 0 invalid
  Pandera:  PASSED

--- corrupted_outliers.csv ---
  Pydantic: 173 valid, 5 invalid
  Pandera:  FAILED

--- corrupted_schema_drift.csv ---
  Pydantic: 0 valid, 178 invalid
  Pandera:  FAILED


### Exercises

1. **Inspect invalid rows**: After calling `validate_with_pydantic(df_corrupted)`, print the `_validation_error` column of the returned invalid DataFrame to see why each row failed.
2. **Compare error messages**: Run the same corrupted file through both validators — which gives more detail about what went wrong?
3. **Performance test**: Use `%%timeit` to compare how long each validator takes on the full clean dataset.
4. **Schema evolution**: Add an extra column to the DataFrame and run `validate_with_pandera()` — does it pass or fail? Check the `strict` parameter in `pa.DataFrameSchema`.
5. **Filter and re-validate**: Use the `valid_df` returned by `validate_with_pydantic()` as input to `validate_with_pandera()`. Does Pandera agree with Pydantic's row-level decisions?
